# SODA-2 end to end

Cite SODA-2 as: Aaron Bansemer. (2026). abansemer/soda2: v2.26.0 (Version v2.26.0) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.18331418

SODA-2 is NCAR's optical array probe software, written in IDL. This package ports it as
a second, independent engine: a different lineage from UIOOPS, with its own sizing, particle
rejection and sample-volume choices. Running both on one raw file gives two answers made
independently, and where they disagree the difference is a property of the methods.

This notebook runs a 2D-S raw file through SODA-2, reads the particle-by-particle output,
turns the protective corrections off to see what they do, and sets the result beside the
v3.5 engine's framing of the same file and beside the truth.

**Before this:** [03_choosing_an_engine](03_choosing_an_engine.ipynb) (Choosing an engine). **Next:** [05_triage_and_gallery](05_triage_and_gallery.ipynb). **Needs:** `pip install "arrayprobe[zarr]"` (xarray).

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/2_processing/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

import shutil
import tempfile

import numpy as np

from arrayprobe.tools import synthetic_truth as st

# A synthetic 2D-S flight from the project's known-truth generator: six seconds of in-focus
# droplets at 50, 100, 150 and 200 um, flown at 130 m/s. The generator records every particle
# it draws, so each engine's count can be checked against the truth.
tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
FLIGHT = st.FlightSpec(n_seconds=6, rate_per_s=12.0, seed=7, depth="in_focus", across="mid",
                       psd=st.PSD(kind="discrete", diameters_um=(50.0, 100.0, 150.0, 200.0)))
SOURCE_RAW = root / "base160101060640.2DS"
truth = st.build_truth_flight(SOURCE_RAW, FLIGHT)
imaged = np.asarray(truth.columns["imaged"])
d_true = np.asarray(truth.columns["d0_um"])[imaged]
print(f"{d_true.size} particles imaged into {SOURCE_RAW.name}")

77 particles imaged into base160101060640.2DS


## The configuration

SODA-2 takes a probe description and processing options, the same fields its IDL takes.
`examples/configs/soda2/2ds_spec.toml` is a 2D-S configuration, every probe field
cited to NCAR's `soda2_probespecs.pro`. The cell keeps it and changes only what names a
file or a time: the raw file, the date, the window and a fixed 130 m/s airspeed in place
of an airspeed record.

In [2]:
cfg_text = Path("examples/configs/soda2/2ds_spec.toml").read_text(encoding="utf-8")
replaced = ("fn", "pth", "date", "starttime", "stoptime", "fixedtas")
keep = [ln for ln in cfg_text.splitlines() if ln.split("=")[0].strip() not in replaced]


def write_config(folder):
    folder.mkdir()
    raw = folder / SOURCE_RAW.name
    shutil.copyfile(SOURCE_RAW, raw)
    header = (f'fn = "{raw.as_posix()}"\ndate = "20160906"\n'
              "starttime = 22000.0\nstoptime = 22006.0\nfixedtas = 130.0\n")
    path = folder / "run.toml"
    path.write_text("\n".join(keep).replace("[soda2]", "[soda2]\n" + header, 1),
                    encoding="utf-8")
    return path


print("\n".join(ln for ln in keep if ln.startswith(("probetype", "res ", "numdiodes",
                                                     "smethod", "eawmethod", "endbins"))))

probetype = "2DS"                                                      # :944
res = 10.0                                                             # :947
numdiodes = 128                                                        # :950
endbins = [5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0, 85.0, 95.0,
smethod = "fastcircle"                                                 # :22
eawmethod = "centerin"                                                 # see note


## Run it

`arrayprobe soda2 run CONFIG --out DIR` on the command line; the cell calls the same entry
point, once with `--protective auto` and once with it `off`. Under `auto`, a probe with no
archived product gets SODA-2's protective corrections: fixes for upstream defects that
destroy data.

In [3]:
import json

from arrayprobe import cli

runs = {}
for mode in ("auto", "off"):
    cfg_path = write_config(root / f"protective_{mode}")
    out = cfg_path.parent / "out"
    cli.main(["soda2", "run", str(cfg_path), "--out", str(out), "--protective", mode])
    pbp_path = next(out.glob("*.pbp.csv"))
    stamp = json.loads(Path(str(pbp_path)[: -len(".csv")] + ".call").read_text(encoding="utf-8"))
    runs[mode] = pbp_path
    print(f"--protective {mode}: corrections applied: "
          f"{stamp['enhancements']['applied'] or 'none'}")

wrote <tmp>\protective_auto\out\20160906_060640_2DS_H.pbp.call
--protective auto: corrections applied: ['dilation_clamp_fix', 'last_buffer_index_fix', 'no_stuck_diode_fix', 'pbp_rejectionflag_ubyte', 'same_time_buffer_fix']
wrote <tmp>\protective_off\out\20160906_060640_2DS_H.pbp.call
--protective off: corrections applied: none


## The particle-by-particle file

One row per particle, with the columns SODA-2 itself writes: time, several sizes
(`diam` from a circle fit, `xsize`, `ysize`, `areasize` and others), shape measures,
edge and depth-of-field flags, and `rejectionflag` (0 accepted; each other code is a
reason, listed in `soda2_reject.pro`).

In [4]:
def read_pbp(path):
    lines = path.read_text(encoding="utf-8").splitlines()
    start = next(i for i, ln in enumerate(lines) if ln.startswith("time,"))
    names = lines[start].split(",")
    rows = np.array([[float(x) for x in ln.split(",")] for ln in lines[start + 1:] if ln],
                    dtype=float)
    return {n: rows[:, i] for i, n in enumerate(names)}


pbp = read_pbp(runs["auto"])
flags, counts = np.unique(pbp["rejectionflag"].astype(int), return_counts=True)
print(f"{pbp['time'].size} particles of {d_true.size} imaged; "
      f"rejection codes: {dict(zip(flags.tolist(), counts.tolist()))}")
print("columns:", ", ".join(pbp))

77 particles of 77 imaged; rejection codes: {0: 77}
columns: time, probetime, buffertime, rawtime, reftime, inttime, diam, xsize, ysize, xextent, oned, twod, areasize, arearatio, arearatiofilled, aspectratio, area, areafilled, perimeterarea, area75, xpos, ypos, allin, centerin, dofflag, edgetouch, sizecorrection, zd, missed, probetas, aircrafttas, overloadflag, particlecounter, orientation, rejectionflag, numregions, diodegaps


## What the protective corrections changed on this file

In [5]:
off = read_pbp(runs["off"])
same_shape = off["time"].size == pbp["time"].size
changed = [n for n in pbp if not same_shape or not np.array_equal(pbp[n], off[n], equal_nan=True)]
print(f"particles: auto {pbp['time'].size}, off {off['time'].size}")
print("columns that differ:", changed or "none")

particles: auto 77, off 77
columns that differ: none


A synthetic file built to be clean reaches none of the defects the corrections repair,
so the two runs agree. On a real file each correction's measured effect is documented in
`enhancements_soda2/`. The stamp records which corrections ran either way, so a result
always says which it is.

## Beside the v3.5 engine, and the truth

The same raw file through v3.5's decode and framing stages. The two engines size
particles differently (SODA-2's `fastcircle` fit against UIOOPS's minimum enclosing circle)
and reject for different reasons, so close agreement is the expectation, not identity.

In [6]:
import sys

import xarray as xr

sys.path.insert(0, "scripts")
import run_flight  # noqa: E402

from arrayprobe.profiles import STOCK_V35  # noqa: E402

run_flight.log = lambda *a, **kw: None
run_flight.use_preset("DEMO_2DS_H")
run_flight.ENGINE, run_flight.PROFILE = "3.5", STOCK_V35
v35 = root / "v35"
v35.mkdir()
raw = v35 / SOURCE_RAW.name
shutil.copyfile(SOURCE_RAW, raw)
run_flight.TAS_FILE = str(st.write_aircraft_tas(v35 / "aircraft.cdf", FLIGHT))
proc = run_flight.stage3(run_flight.stage2(raw, v35), v35)
with xr.open_dataset(proc, mask_and_scale=False, decode_times=False) as ds:
    d_v35 = np.asarray(ds["image_diam_minR"].values, float).ravel() * 1000.0     # mm -> um
    code = np.asarray(ds["image_auto_reject"].values).ravel().astype(int)
ok_v35 = np.isin(code, [ord(c) for c in "0HhuR"])
d_soda = pbp["diam"][pbp["rejectionflag"] == 0]

print(f"{'':8s} {'accepted':>9s}   " + "  ".join(f"{d:>5.0f}um" for d in (50, 100, 150, 200)))
for name, d in (("truth", d_true), ("SODA-2", d_soda), ("v3.5", d_v35[ok_v35])):
    near = [int(np.sum(np.abs(d - c) <= 15)) for c in (50, 100, 150, 200)]
    print(f"{name:8s} {d.size:9d}   " + "  ".join(f"{n:7d}" for n in near))

          accepted      50um    100um    150um    200um
truth           77        22       17       17       21
SODA-2          77        22       17       17       21
v3.5            77        22       17       17       21


Each column counts the accepted particles sized within 15 um (1.5 pixels) of one true
diameter.

## Next

- `08_opt_ins.ipynb`: every SODA-2 correction that is off by default, and how to turn it on.
- [`01_cip_vs_2ds.ipynb`](../7_analysis_and_outputs/01_cip_vs_2ds.ipynb): two different probes on one common size grid.
- `docs/developer/UPSTREAM_REVIEW.md`: the port's review of the SODA-2 and UIOOPS source,
  and where it is kept.